# PrimeVul matched Qwen2.5-Coder-14B locked-test baseline

This notebook runs the larger open-source model in NF4 4-bit mode on the same 24,008-record leakage-controlled natural test protocol used by the Thesis. It is a quantized model baseline, not the Phoenix paired experiment. A T4 GPU is supported.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes safetensors

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess, sys

ROOT = Path('/content/drive/MyDrive/primevul')
SCRIPT = ROOT / 'run_qwen_colab.py'
DATA = ROOT / 'primevul_test.jsonl'
MANIFEST = ROOT / 'thesis_primevul_leakage_controlled_manifest.json'
OUTPUT = ROOT / 'qwen14b_4bit_test_results.jsonl'
MODEL = 'Qwen/Qwen2.5-Coder-14B-Instruct'
SAMPLE_COUNT = 0  # 3 for smoke test; 0 for all retained records

for path in (SCRIPT, DATA, MANIFEST):
    if not path.exists():
        raise FileNotFoundError(path)

command = [
    sys.executable, str(SCRIPT),
    '--data', str(DATA),
    '--manifest', str(MANIFEST),
    '--partition', 'test',
    '--model', MODEL,
    '--revision', 'aedcc2d42b622764e023cf882b6652e646b95671',
    '--quantization', '4bit',
    '--limit', str(SAMPLE_COUNT),
    '--dtype', 'auto',
    '--device-map', 'auto',
    '--max-new-tokens', '16',
    '--progress-every', '25',
    '--output', str(OUTPUT),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json
meta = json.loads(Path(str(OUTPUT) + '.meta.json').read_text())
print(json.dumps(meta, indent=2))
assert meta['status'] == 'complete'
print('Completed:', meta['run']['records_processed'], 'records')

## Evaluation

Copy `evaluate_qwen_matched_baseline.py` into the same Drive folder and run it from a Colab cell, or evaluate the output on the thesis workstation. Unparseable outputs must remain separately reported.

In [ ]:
EVALUATOR = ROOT / 'evaluate_qwen_matched_baseline.py'
STATS = ROOT / 'qwen14b_4bit_test_baseline_statistics.json'
if EVALUATOR.exists():
    subprocess.run([
        sys.executable, str(EVALUATOR),
        '--input', str(OUTPUT),
        '--output', str(STATS),
        '--model', MODEL,
    ], check=True)
else:
    print('Upload evaluate_qwen_matched_baseline.py to run automatic evaluation.')